# Agent System

An AI agent that plans, executes tools, stores memory, and reflects using Sarvam `sarvam-m`.

## Architecture

```
User Query
        |
        v
plan_task()         <- sarvam-m produces a JSON step list
        |
        v
execute_step()      <- routes each step to a tool
        |               calculator  (safe arithmetic)
        |               search      (keyword knowledge base)
        |               notes       (in-memory key/value store)
        v
MemoryStore         <- accumulates step results
        |
        v
reflect()           <- sarvam-m synthesises the final answer
        |
        v
Final Answer
```

In [ ]:
%pip install sarvamai>=0.1.26 python-dotenv>=1.0.0

## Setup

Load environment variables, initialise the Sarvam AI client, and add the recipe root to
`sys.path` so the `tools/` modules are importable.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

# Add recipe root to sys.path so tools/ is importable
_RECIPE_ROOT = str(Path.cwd())
if _RECIPE_ROOT not in sys.path:
    sys.path.insert(0, _RECIPE_ROOT)

from tools.calculator import calculate
from tools.search import search
from tools.notes import save_note, read_note, list_notes

print("Setup complete.")

## Memory Store

`MemoryStore` accumulates step descriptions and their results across the agent loop.
Each entry records the step number, description, tool used, and tool output.

In [ ]:
class MemoryStore:
    """Accumulate step results across an agent run.

    Each entry is a dict with keys: step, description, tool, result.
    """

    def __init__(self) -> None:
        self._entries: list[dict[str, Any]] = []

    def add(self, step: int, description: str, tool: str, result: str) -> None:
        """Append a completed step to memory."""
        self._entries.append(
            {"step": step, "description": description, "tool": tool, "result": result}
        )

    def as_text(self) -> str:
        """Format all entries as a readable string for LLM prompts."""
        if not self._entries:
            return "No steps completed yet."
        lines = []
        for entry in self._entries:
            lines.append(
                f"Step {entry['step']} [{entry['tool']}]: {entry['description']}\n"
                f"  Result: {entry['result']}"
            )
        return "\n\n".join(lines)

    def __len__(self) -> int:
        return len(self._entries)

## Planner

`plan_task` sends the user query to `sarvam-m` with a structured prompt that requests a
JSON step list. Each step specifies which tool to call and what input to pass.

Supported tools: `calculator`, `search`, `notes`, `none`.

In [ ]:
_PLAN_SYSTEM_PROMPT = """\
You are a planning assistant. Given a user query, break it into up to 5 concrete steps.

Return ONLY a valid JSON array. Each element must be an object with these exact keys:
  "step"        : integer, starting at 1
  "description" : short description of what this step does
  "tool"        : one of "calculator", "search", "notes", "none"
  "input"       : the exact string to pass to the tool, or "" for "none"

Rules:
- Use "calculator" for arithmetic expressions only (e.g. "15 / 100 * 240")
- Use "search" for factual lookups (e.g. "Sarvam AI")
- Use "notes" for saving or reading stored information (prefix input with "save:<key>:<content>" or "read:<key>" or "list")
- Use "none" when the step requires no external tool
- Do not include any text outside the JSON array
"""


def plan_task(query: str, sarvam_client: SarvamAI = client) -> list[dict[str, Any]]:
    """Break a user query into a step-by-step execution plan.

    Args:
        query: The user's task or question.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        List of step dicts with keys: step, description, tool, input.
        Falls back to a single none-tool step on JSON parse failure.
    """
    messages = [
        {"role": "system", "content": _PLAN_SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]
    response = sarvam_client.chat.completions(messages=messages, model="sarvam-m")
    raw = response.choices[0].message.content.strip()

    try:
        plan = json.loads(raw)
        if isinstance(plan, list):
            return plan
    except json.JSONDecodeError:
        pass

    # Fallback: single step with no tool
    return [{"step": 1, "description": query, "tool": "none", "input": query}]

## Tool Execution

`execute_step` reads the `tool` field of a plan step and calls the appropriate function.
For the `notes` tool, the input is parsed to distinguish save / read / list operations.

In [ ]:
def execute_step(step: dict[str, Any], memory: MemoryStore) -> str:
    """Execute one plan step by routing to the appropriate tool.

    Args:
        step: A plan step dict with keys: step, description, tool, input.
        memory: The running MemoryStore (passed so notes can query prior results).

    Returns:
        Tool output string.
    """
    tool = step.get("tool", "none")
    tool_input = step.get("input", "")

    if tool == "calculator":
        result = calculate(tool_input)

    elif tool == "search":
        result = search(tool_input)

    elif tool == "notes":
        if tool_input.startswith("save:"):
            parts = tool_input[5:].split(":", 1)
            if len(parts) == 2:
                result = save_note(parts[0], parts[1])
            else:
                result = "Error: notes save format is 'save:<key>:<content>'"
        elif tool_input.startswith("read:"):
            result = read_note(tool_input[5:])
        elif tool_input == "list":
            result = list_notes()
        else:
            result = f"Unknown notes operation: {tool_input!r}"

    else:
        result = f"No tool used. Step: {step.get('description', '')}"

    memory.add(
        step=step.get("step", 0),
        description=step.get("description", ""),
        tool=tool,
        result=result,
    )
    return result

## Reflection

`reflect` sends the original query and all step results to `sarvam-m`.
The model synthesises a coherent final answer from the accumulated evidence.

In [ ]:
_REFLECT_SYSTEM_PROMPT = """\
You are a synthesis assistant. You will receive a user query and a list of completed
agent steps with their results. Write a clear, concise final answer that directly
addresses the query using the step results as evidence.

Do not repeat the step list verbatim. Integrate the results into a natural response.
"""


def reflect(
    query: str,
    memory: MemoryStore,
    sarvam_client: SarvamAI = client,
) -> str:
    """Synthesise a final answer from the accumulated step results.

    Args:
        query: The original user query.
        memory: MemoryStore populated by the agent loop.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Final answer string from sarvam-m.
    """
    user_message = (
        f"Query: {query}\n\n"
        f"Completed steps:\n{memory.as_text()}"
    )
    messages = [
        {"role": "system", "content": _REFLECT_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    response = sarvam_client.chat.completions(messages=messages, model="sarvam-m")
    return response.choices[0].message.content.strip()

## Agent Loop

`run_agent` orchestrates the full cycle:
1. Plan the query into steps.
2. Execute each step and store results in memory.
3. Reflect over all results to produce the final answer.

In [ ]:
def run_agent(
    query: str,
    sarvam_client: SarvamAI = client,
) -> dict[str, Any]:
    """Run the full agent loop: plan, execute, reflect.

    Args:
        query: User task or question.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Dict with keys: query, plan, step_results, final_answer, memory_snapshot.
    """
    memory = MemoryStore()

    # Phase 1: Plan
    plan = plan_task(query, sarvam_client)

    # Phase 2: Execute
    step_results: list[dict[str, Any]] = []
    for step in plan:
        result = execute_step(step, memory)
        step_results.append({"step": step, "result": result})

    # Phase 3: Reflect
    final_answer = reflect(query, memory, sarvam_client)

    return {
        "query": query,
        "plan": plan,
        "step_results": step_results,
        "final_answer": final_answer,
        "memory_snapshot": memory.as_text(),
    }

## Demo

Run the agent on a query that exercises both the calculator and search tools.

In [ ]:
result = run_agent(
    "What is 15 percent of 240, and what is Sarvam AI?",
    sarvam_client=client,
)

print("Query       :", result["query"])
print()
print("Plan        :", json.dumps(result["plan"], indent=2))
print()
print("Step results:")
for sr in result["step_results"]:
    print(f"  Step {sr['step'].get('step')}: {sr['result']}")
print()
print("Final answer:", result["final_answer"])

### Notes Tool Demo

This demo saves a fact to the notes store then retrieves it in a subsequent step.

In [ ]:
result2 = run_agent(
    "Save the note 'capital_of_india' as 'New Delhi', then read it back and confirm.",
    sarvam_client=client,
)

print("Query       :", result2["query"])
print()
print("Memory snapshot:")
print(result2["memory_snapshot"])
print()
print("Final answer:", result2["final_answer"])

## Error Reference and Resources

### Common Errors

| Error | Cause | Fix |
|:------|:------|:----|
| `RuntimeError: SARVAM_API_KEY is not set` | Missing `.env` | Copy `.env.example` to `.env` and add your key |
| `sarvamai.APIStatusError: 401` | Invalid API key | Verify key at dashboard.sarvam.ai |
| `sarvamai.APIStatusError: 429` | Rate limit exceeded | Add a short delay between agent calls |
| `ModuleNotFoundError: tools` | `sys.path` not set | Ensure cell 3 ran successfully before other cells |
| `json.JSONDecodeError` | Planner returned non-JSON | Already handled — falls back to single `none` step |

### Resources

- [Sarvam AI Documentation](https://docs.sarvam.ai/)
- [Sarvam API Dashboard](https://dashboard.sarvam.ai/)